# Project Arbitrary Persona Vector to Assistant Axis Landscape

**Interactive tool for visualizing persona vectors in the Assistant Axis PCA space**

This notebook provides modular functions to project any persona vector into the Assistant Axis landscape and visualize its position relative to:
- 275 character role vectors
- Default assistant vector
- The Assistant Axis direction

Uses interactive Plotly visualizations with hover labels for exploration.

In [1]:
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import torch
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA
from huggingface_hub import snapshot_download

# Suppress HF progress bars
from huggingface_hub.utils import disable_progress_bars
disable_progress_bars()

/Users/irakl/Desktop/Projects/LASR/persona-shattering-lasr/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

In [2]:
# MODEL_NAME: Determines which pre-computed vectors to download from HuggingFace
# Options: "gemma-2-27b", "qwen-3-32b", "llama-3.3-70b"
MODEL_NAME = "gemma-2-27b"

# TARGET_LAYER: Which layer's activations to analyze
# Layer 22 for Gemma-2-27b (middle layer, empirically best for persona representations)
TARGET_LAYER = 22

# HuggingFace dataset
REPO_ID = "lu-christina/assistant-axis-vectors"

print(f"Model: {MODEL_NAME}")
print(f"Target layer: {TARGET_LAYER}")

Model: gemma-2-27b
Target layer: 22


## Helper Classes and Functions

In [3]:
class MeanScaler:
    """Mean centering scaler (following assistant-axis implementation)."""
    def __init__(self):
        self.mean = None
    
    def fit(self, X):
        self.mean = np.mean(X, axis=0)
        return self
    
    def transform(self, X):
        return X - self.mean
    
    def fit_transform(self, X):
        return self.fit(X).transform(X)


def load_vectors_from_hf(
    model_name: str,
    repo_id: str = "lu-christina/assistant-axis-vectors"
) -> Tuple[Dict[str, torch.Tensor], torch.Tensor, torch.Tensor]:
    """Load role vectors, default vector, and assistant axis from HuggingFace.
    
    Args:
        model_name: Model identifier (e.g., "gemma-2-27b")
        repo_id: HuggingFace dataset repository
    
    Returns:
        Tuple of (role_vectors_dict, default_vector, assistant_axis)
        - role_vectors_dict: {role_name: tensor of shape [n_layers, hidden_size]}
        - default_vector: tensor of shape [n_layers, hidden_size]
        - assistant_axis: tensor of shape [n_layers, hidden_size]
    """
    print(f"Downloading vectors from {repo_id}...")
    local_dir = snapshot_download(
        repo_id=repo_id,
        repo_type="dataset",
        allow_patterns=[
            f"{model_name}/role_vectors/*.pt",
            f"{model_name}/default_vector.pt",
            f"{model_name}/assistant_axis.pt",
        ]
    )
    
    # Load role vectors
    role_vectors_dir = Path(local_dir) / model_name / "role_vectors"
    role_vectors = {}
    for pt_file in sorted(role_vectors_dir.glob("*.pt")):
        role_name = pt_file.stem
        role_vectors[role_name] = torch.load(pt_file, map_location="cpu", weights_only=False)
    
    # Load default vector
    default_vector_path = Path(local_dir) / model_name / "default_vector.pt"
    default_vector = torch.load(default_vector_path, map_location="cpu", weights_only=False)
    
    # Load assistant axis
    assistant_axis_path = Path(local_dir) / model_name / "assistant_axis.pt"
    assistant_axis = torch.load(assistant_axis_path, map_location="cpu", weights_only=False)
    
    print(f"Loaded {len(role_vectors)} role vectors")
    print(f"Default vector shape: {default_vector.shape}")
    print(f"Assistant axis shape: {assistant_axis.shape}")
    
    return role_vectors, default_vector, assistant_axis


def fit_pca_landscape(
    role_vectors: Dict[str, torch.Tensor],
    target_layer: int
) -> Tuple[PCA, MeanScaler, np.ndarray, List[str], np.ndarray]:
    """Fit PCA on role vectors at target layer.
    
    Args:
        role_vectors: Dict of role vectors
        target_layer: Layer index to use
    
    Returns:
        Tuple of (pca, scaler, pca_result, role_labels, variance_explained)
    """
    # Extract vectors at target layer
    role_labels = list(role_vectors.keys())
    role_vectors_array = torch.stack(
        [role_vectors[label][target_layer] for label in role_labels]
    ).float().numpy()
    
    print(f"Fitting PCA on {len(role_labels)} vectors at layer {target_layer}...")
    
    # Fit scaler and PCA
    scaler = MeanScaler()
    role_vectors_scaled = scaler.fit_transform(role_vectors_array)
    
    pca = PCA()
    pca_result = pca.fit_transform(role_vectors_scaled)
    
    variance_explained = pca.explained_variance_ratio_
    cumulative_variance = np.cumsum(variance_explained)
    
    print(f"PCA fitted with {len(variance_explained)} components")
    print(f"Variance explained by first 3 PCs: {variance_explained[:3]}")
    print(f"Cumulative variance (first 3 PCs): {cumulative_variance[2]:.4f}")
    
    return pca, scaler, pca_result, role_labels, variance_explained


def project_vector(
    vector: torch.Tensor,
    scaler: MeanScaler,
    pca: PCA
) -> np.ndarray:
    """Project a vector into PC space.
    
    Args:
        vector: 1D tensor or numpy array
        scaler: Fitted MeanScaler
        pca: Fitted PCA
    
    Returns:
        PC coordinates (1D array)
    """
    if isinstance(vector, torch.Tensor):
        vector = vector.float().numpy()
    
    vector_scaled = scaler.transform(vector.reshape(1, -1))
    vector_pc = pca.transform(vector_scaled)[0]
    
    return vector_pc


def create_test_vector(
    source_vector: torch.Tensor,
    noise_scale: float = 0.1,
    all_vectors: Optional[torch.Tensor] = None
) -> torch.Tensor:
    """Create a noisy test vector from a source vector.
    
    Args:
        source_vector: Source vector to add noise to
        noise_scale: Scale of noise relative to std of all vectors
        all_vectors: All role vectors (used to compute std). If None, uses source vector std.
    
    Returns:
        Noisy vector
    """
    if all_vectors is not None:
        vector_std = all_vectors.std()
    else:
        vector_std = source_vector.std()
    
    noise = torch.randn_like(source_vector) * noise_scale * vector_std
    noisy_vector = source_vector + noise
    
    return noisy_vector

## Visualization Functions

In [4]:
def plot_2d_landscape(
    pca_result: np.ndarray,
    role_labels: List[str],
    variance_explained: np.ndarray,
    default_pc: np.ndarray,
    assistant_axis_pc: Optional[np.ndarray] = None,
    test_vectors: Optional[Dict[str, np.ndarray]] = None,
    title: str = "Persona Vectors in PC Space"
) -> go.Figure:
    """Create interactive 2D scatter plot (PC1 vs PC2) with hover labels.
    
    Args:
        pca_result: PC coordinates for all role vectors (N x K)
        role_labels: List of role names
        variance_explained: Variance explained by each PC
        default_pc: PC coordinates for default assistant
        assistant_axis_pc: Optional PC coordinates for assistant axis direction
        test_vectors: Optional dict of {name: pc_coordinates} for test vectors
        title: Plot title
    
    Returns:
        Plotly figure
    """
    fig = go.Figure()
    
    # Plot all role vectors with hover labels
    fig.add_trace(go.Scatter(
        x=pca_result[:, 0],
        y=pca_result[:, 1],
        mode='markers',
        marker=dict(size=6, color='lightgray', opacity=0.6),
        text=role_labels,
        hovertemplate='<b>%{text}</b><br>PC1: %{x:.2f}<br>PC2: %{y:.2f}<extra></extra>',
        name='Role vectors',
        showlegend=True
    ))
    
    # Plot default assistant
    fig.add_trace(go.Scatter(
        x=[default_pc[0]],
        y=[default_pc[1]],
        mode='markers',
        marker=dict(size=20, color='blue', symbol='star', line=dict(color='black', width=2)),
        text=['Default Assistant'],
        hovertemplate='<b>%{text}</b><br>PC1: %{x:.2f}<br>PC2: %{y:.2f}<extra></extra>',
        name='Default Assistant',
        showlegend=True
    ))
    
    # Plot assistant axis if provided
    if assistant_axis_pc is not None:
        # Draw arrow from origin to assistant axis direction
        # Scale for visibility
        scale = 0.3 * max(np.abs(pca_result[:, :2]).max(), np.abs(default_pc[:2]).max())
        axis_end = assistant_axis_pc[:2] / np.linalg.norm(assistant_axis_pc[:2]) * scale
        
        fig.add_trace(go.Scatter(
            x=[0, axis_end[0]],
            y=[0, axis_end[1]],
            mode='lines+markers',
            line=dict(color='red', width=3),
            marker=dict(size=[0, 12], color='red', symbol='arrow-up'),
            text=['', 'Assistant Axis'],
            hovertemplate='<b>Assistant Axis</b><br>Direction in PC space<extra></extra>',
            name='Assistant Axis',
            showlegend=True
        ))
    
    # Plot test vectors if provided
    if test_vectors:
        colors = ['green', 'orange', 'purple', 'cyan', 'magenta']
        for i, (name, pc_coords) in enumerate(test_vectors.items()):
            color = colors[i % len(colors)]
            fig.add_trace(go.Scatter(
                x=[pc_coords[0]],
                y=[pc_coords[1]],
                mode='markers',
                marker=dict(size=15, color=color, symbol='x', line=dict(color='black', width=2)),
                text=[name],
                hovertemplate='<b>%{text}</b><br>PC1: %{x:.2f}<br>PC2: %{y:.2f}<extra></extra>',
                name=name,
                showlegend=True
            ))
    
    fig.update_layout(
        title=title,
        xaxis_title=f'PC1 ({variance_explained[0]:.1%} variance)',
        yaxis_title=f'PC2 ({variance_explained[1]:.1%} variance)',
        hovermode='closest',
        width=900,
        height=700,
        template='plotly_white'
    )
    
    return fig


def plot_3d_landscape(
    pca_result: np.ndarray,
    role_labels: List[str],
    variance_explained: np.ndarray,
    default_pc: np.ndarray,
    assistant_axis_pc: Optional[np.ndarray] = None,
    test_vectors: Optional[Dict[str, np.ndarray]] = None,
    title: str = "Persona Vectors in 3D PC Space"
) -> go.Figure:
    """Create interactive 3D scatter plot with hover labels.
    
    Args:
        pca_result: PC coordinates for all role vectors (N x K)
        role_labels: List of role names
        variance_explained: Variance explained by each PC
        default_pc: PC coordinates for default assistant
        assistant_axis_pc: Optional PC coordinates for assistant axis direction
        test_vectors: Optional dict of {name: pc_coordinates} for test vectors
        title: Plot title
    
    Returns:
        Plotly figure
    """
    fig = go.Figure()
    
    # Plot all role vectors
    fig.add_trace(go.Scatter3d(
        x=pca_result[:, 0],
        y=pca_result[:, 1],
        z=pca_result[:, 2],
        mode='markers',
        marker=dict(size=4, color='lightgray', opacity=0.5),
        text=role_labels,
        hovertemplate='<b>%{text}</b><br>PC1: %{x:.2f}<br>PC2: %{y:.2f}<br>PC3: %{z:.2f}<extra></extra>',
        name='Role vectors'
    ))
    
    # Plot default assistant
    fig.add_trace(go.Scatter3d(
        x=[default_pc[0]],
        y=[default_pc[1]],
        z=[default_pc[2]],
        mode='markers',
        marker=dict(size=12, color='blue', symbol='diamond'),
        text=['Default Assistant'],
        hovertemplate='<b>%{text}</b><br>PC1: %{x:.2f}<br>PC2: %{y:.2f}<br>PC3: %{z:.2f}<extra></extra>',
        name='Default Assistant'
    ))
    
    # Plot assistant axis if provided
    if assistant_axis_pc is not None:
        scale = 0.3 * max(np.abs(pca_result[:, :3]).max(), np.abs(default_pc[:3]).max())
        axis_end = assistant_axis_pc[:3] / np.linalg.norm(assistant_axis_pc[:3]) * scale
        
        fig.add_trace(go.Scatter3d(
            x=[0, axis_end[0]],
            y=[0, axis_end[1]],
            z=[0, axis_end[2]],
            mode='lines+markers',
            line=dict(color='red', width=6),
            marker=dict(size=[0, 8], color='red'),
            text=['', 'Assistant Axis'],
            hovertemplate='<b>Assistant Axis</b><extra></extra>',
            name='Assistant Axis'
        ))
    
    # Plot test vectors if provided
    if test_vectors:
        colors = ['green', 'orange', 'purple', 'cyan', 'magenta']
        for i, (name, pc_coords) in enumerate(test_vectors.items()):
            color = colors[i % len(colors)]
            fig.add_trace(go.Scatter3d(
                x=[pc_coords[0]],
                y=[pc_coords[1]],
                z=[pc_coords[2]],
                mode='markers',
                marker=dict(size=10, color=color, symbol='x'),
                text=[name],
                hovertemplate='<b>%{text}</b><br>PC1: %{x:.2f}<br>PC2: %{y:.2f}<br>PC3: %{z:.2f}<extra></extra>',
                name=name
            ))
    
    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title=f'PC1 ({variance_explained[0]:.1%})',
            yaxis_title=f'PC2 ({variance_explained[1]:.1%})',
            zaxis_title=f'PC3 ({variance_explained[2]:.1%})'
        ),
        width=900,
        height=700,
        template='plotly_white'
    )
    
    return fig

## Load Data and Fit PCA

In [5]:
# Load vectors from HuggingFace
role_vectors, default_vector, assistant_axis = load_vectors_from_hf(MODEL_NAME, REPO_ID)

Loaded 275 role vectors
Default vector shape: torch.Size([46, 4608])
Assistant axis shape: torch.Size([46, 4608])


In [6]:
# Fit PCA on role vectors
pca, scaler, pca_result, role_labels, variance_explained = fit_pca_landscape(
    role_vectors, TARGET_LAYER
)

Fitting PCA on 275 vectors at layer 22...
PCA fitted with 275 components
Variance explained by first 3 PCs: [0.4880166  0.09798774 0.06981188]
Cumulative variance (first 3 PCs): 0.6558


In [7]:
# Project special vectors into PC space
default_pc = project_vector(default_vector[TARGET_LAYER], scaler, pca)
assistant_axis_pc = project_vector(assistant_axis[TARGET_LAYER], scaler, pca)

print(f"Default assistant PC coordinates (first 3): {default_pc[:3]}")
print(f"Assistant axis PC coordinates (first 3): {assistant_axis_pc[:3]}")

Default assistant PC coordinates (first 3): [674.28394  -21.292511  65.77762 ]
Assistant axis PC coordinates (first 3): [-10189.124   -5145.0283   -869.7756]


## Create Test Vector

For demonstration, we'll create a noisy version of a source persona to test the projection.

In [8]:
# Select source persona and create noisy test vector
SOURCE_PERSONA = "pirate"  # Change to any role name
NOISE_SCALE = 0.1

if SOURCE_PERSONA not in role_vectors:
    print(f"Available roles: {sorted(role_vectors.keys())[:20]}...")
    raise ValueError(f"Role '{SOURCE_PERSONA}' not found")

# Get source vector and create noisy version
source_vector_at_layer = role_vectors[SOURCE_PERSONA][TARGET_LAYER]
all_vectors_at_layer = torch.stack([v[TARGET_LAYER] for v in role_vectors.values()]).float()

noisy_test_vector = create_test_vector(
    source_vector_at_layer,
    noise_scale=NOISE_SCALE,
    all_vectors=all_vectors_at_layer
)

# Project both into PC space
source_pc = project_vector(source_vector_at_layer, scaler, pca)
noisy_pc = project_vector(noisy_test_vector, scaler, pca)

# Compute distance
pc_distance = np.linalg.norm(source_pc[:3] - noisy_pc[:3])
original_distance = (source_vector_at_layer - noisy_test_vector).norm().item()

print(f"Source persona: {SOURCE_PERSONA}")
print(f"Original space distance: {original_distance:.4f}")
print(f"PC space distance (first 3 PCs): {pc_distance:.4f}")
print(f"Source PC: {source_pc[:3]}")
print(f"Noisy PC: {noisy_pc[:3]}")

Source persona: pirate
Original space distance: 1568.0000
PC space distance (first 3 PCs): 16.2255
Source PC: [-1114.2961   -379.22385   -81.8283 ]
Noisy PC: [-1123.6812   -369.5892    -90.90357]


## Visualize: 2D Interactive Plot

In [9]:
# Create test vectors dict for plotting
test_vectors = {
    f"Source ({SOURCE_PERSONA})": source_pc,
    "Test Vector (noisy)": noisy_pc,
}

fig_2d = plot_2d_landscape(
    pca_result=pca_result,
    role_labels=role_labels,
    variance_explained=variance_explained,
    default_pc=default_pc,
    assistant_axis_pc=assistant_axis_pc,
    test_vectors=test_vectors,
    title="Persona Vectors in Assistant Axis Landscape (2D)"
)

fig_2d.show()

## Visualize: 3D Interactive Plot

In [10]:
fig_3d = plot_3d_landscape(
    pca_result=pca_result,
    role_labels=role_labels,
    variance_explained=variance_explained,
    default_pc=default_pc,
    assistant_axis_pc=assistant_axis_pc,
    test_vectors=test_vectors,
    title="Persona Vectors in Assistant Axis Landscape (3D)"
)

fig_3d.show()

## Project Your Own Vector

Use this cell to project any arbitrary persona vector into the landscape.

**Expected input format:**
- `your_vector`: 1D tensor or numpy array of shape `[hidden_size]` at the target layer
- For Gemma-2-27b layer 22: shape should be `[4608]`

In [ ]:
# Example: Project a custom vector
# Replace this with your actual persona vector from fine-tuning

# For demo, we'll use another random role vector
CUSTOM_PERSONA_NAME = "detective"  # Change this
your_vector = role_vectors[CUSTOM_PERSONA_NAME][TARGET_LAYER]

# Project to PC space
your_pc = project_vector(your_vector, scaler, pca)

print(f"Custom vector '{CUSTOM_PERSONA_NAME}' projected to PC space:")
print(f"PC coordinates (first 3): {your_pc[:3]}")

# Visualize
custom_test_vectors = {
    f"Custom: {CUSTOM_PERSONA_NAME}": your_pc,
}

fig_custom = plot_2d_landscape(
    pca_result=pca_result,
    role_labels=role_labels,
    variance_explained=variance_explained,
    default_pc=default_pc,
    assistant_axis_pc=assistant_axis_pc,
    test_vectors=custom_test_vectors,
    title=f"Custom Persona: {CUSTOM_PERSONA_NAME}"
)

fig_custom.show()

Custom vector 'detective' projected to PC space:
PC coordinates (first 3): [ 570.32214 -134.97226 -157.90665]


## Summary

This notebook provides a reusable toolkit for projecting persona vectors into the Assistant Axis landscape:

✅ **Modular functions** - Easy to use with any vector  
✅ **Interactive visualizations** - Hover to see role names  
✅ **Assistant Axis included** - Shows the main persona direction  
✅ **Validated workflow** - Ready for fine-tuned persona vectors  

**Next steps:**
1. Fine-tune avoid-letter-o persona
2. Generate rollouts and extract activations
3. Compute persona vector at layer 22
4. Load it here and project using `project_vector()`
5. Visualize where it lands in the landscape!